<a href="https://colab.research.google.com/github/AnuragTiwari-hub/AIML/blob/main/AIML12(PCA).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving data.csv to data.csv


In [ ]:
# ==========================================
# BLOCK 1: IMPORTING LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

file_path = '/content/data.csv'

# ==========================================
# BLOCK 2: EXPERIMENT 4 (IMPORT, EXPLORE, EXPORT)
# ==========================================
print("--- EXPERIMENT 4: DATA EXPLORATION ---")

try:
    df = pd.read_csv(file_path)
    print("Dataset imported successfully!\n")
except FileNotFoundError:
    print(f"Error: Could not find file at {file_path}. Please upload it.")

print(f"Number of Rows: {df.shape[0]}")
print(f"Number of Columns: {df.shape[1]}")
print(f"Total Size (Elements): {df.size}")

print("\n--- First Five Rows ---")
display(df.head())

print("\n--- Number of Missing Values ---")
print(df.isnull().sum())

num_cols = df.select_dtypes(include=[np.number])

print("\n--- Summary of Numerical Columns ---")
print("\nSum:\n", num_cols.sum())
print("\nAverage, Min, and Max:\n", num_cols.agg(['mean', 'min', 'max']).T)

export_path = '/content/exported_dataset.csv'
df.to_csv(export_path, index=False)
print(f"\nData exported to: {export_path}")

# ==========================================
# BLOCK 3: EXPERIMENT 6 (PREPROCESSING)
# ==========================================
print("\n\n--- EXPERIMENT 6: DATA PREPROCESSING ---")

# FAILSAFE 1: Drop columns that are entirely missing (100% NaNs)
df = df.dropna(axis=1, how='all')

# Re-select numerical and categorical columns in case we dropped some
num_cols = df.select_dtypes(include=[np.number])
cat_cols = df.select_dtypes(exclude=[np.number])

print("Handling missing values...")

for col in num_cols.columns:
    df[col] = df[col].fillna(df[col].mean())

for col in cat_cols.columns:
    if not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after imputation:\n", df.isnull().sum())

print("\nHandling outliers (capping method)...")

def handle_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    dataframe[column] = np.clip(dataframe[column], lower_bound, upper_bound)
    return dataframe

for col in num_cols.columns:
    df = handle_outliers_iqr(df, col)

print("Outliers capped to the IQR lower and upper bounds.")

# ==========================================
# BLOCK 4: EXPERIMENT 8 (PCA)
# ==========================================
print("\n\n--- EXPERIMENT 8: PRINCIPAL COMPONENT ANALYSIS (PCA) ---")

X = df.select_dtypes(include=[np.number])

# FAILSAFE 2: Drop columns with zero variance (where all values are identical)
# This prevents the StandardScaler from dividing by zero and creating NaNs!
X = X.loc[:, X.nunique() > 1]

# FAILSAFE 3: Drop any remaining rows that still somehow have NaNs
X = X.dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_components = min(2, X.shape[1])
pca = PCA(n_components=n_components)

X_pca = pca.fit_transform(X_scaled)

pca_columns = [f'Principal Component {i+1}' for i in range(n_components)]
pca_df = pd.DataFrame(data=X_pca, columns=pca_columns)

print("Original dataset shape:", X.shape)
print("Reduced dataset shape (after PCA):", pca_df.shape)

print("\n--- Explained Variance Ratio ---")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"Component {i+1}: {var * 100:.2f}%")

print("\n--- First 5 rows of PCA Transformed Data ---")
display(pca_df.head())

--- EXPERIMENT 4: DATA EXPLORATION ---
Dataset imported successfully!

Number of Rows: 569
Number of Columns: 33
Total Size (Elements): 18777

--- First Five Rows ---


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN



--- Number of Missing Values ---
id                           0
diagnosis                    0
radius_mean                  0
texture_mean                 0
perimeter_mean               0
area_mean                    0
smoothness_mean              0
compactness_mean             0
concavity_mean               0
concave points_mean          0
symmetry_mean                0
fractal_dimension_mean       0
radius_se                    0
texture_se                   0
perimeter_se                 0
area_se                      0
smoothness_se                0
compactness_se               0
concavity_se                 0
concave points_se            0
symmetry_se                  0
fractal_dimension_se         0
radius_worst                 0
texture_worst                0
perimeter_worst              0
area_worst                   0
smoothness_worst             0
compactness_worst            0
concavity_worst              0
concave points_worst         0
symmetry_worst               0
fract

,Principal Component 1,Principal Component 2
0,9.246843,2.319114
1,2.890598,-4.266495
2,6.715371,-1.367290
3,5.716746,7.263435
4,4.904217,-2.126864


In [ ]:
# BLOCK 4: EXPERIMENT 8 (PCA)
# ==========================================
print("\n\n--- EXPERIMENT 8: PRINCIPAL COMPONENT ANALYSIS (PCA) ---")

X = df.select_dtypes(include=[np.number])

# FAILSAFE 2: Drop columns with zero variance (where all values are identical)
# This prevents the StandardScaler from dividing by zero and creating NaNs!
X = X.loc[:, X.nunique() > 1]

# FAILSAFE 3: Drop any remaining rows that still somehow have NaNs
X = X.dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_components = min(2, X.shape[1])
pca = PCA(n_components=n_components)

X_pca = pca.fit_transform(X_scaled)

pca_columns = [f'Principal Component {i+1}' for i in range(n_components)]
pca_df = pd.DataFrame(data=X_pca, columns=pca_columns)

print("Original dataset shape:", X.shape)
print("Reduced dataset shape (after PCA):", pca_df.shape)

print("\n--- Explained Variance Ratio ---")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"Component {i+1}: {var * 100:.2f}%")

print("\n--- First 5 rows of PCA Transformed Data ---")
display(pca_df.head())



--- EXPERIMENT 8: PRINCIPAL COMPONENT ANALYSIS (PCA) ---
Original dataset shape: (569, 31)
Reduced dataset shape (after PCA): (569, 2)

--- Explained Variance Ratio ---
Component 1: 45.38%
Component 2: 18.17%

--- First 5 rows of PCA Transformed Data ---


,Principal Component 1,Principal Component 2
0,9.246843,2.319114
1,2.890598,-4.266495
2,6.715371,-1.367290
3,5.716746,7.263435
4,4.904217,-2.126864
